# OCOM5200M: Machine Learning - Unit 3 - Activity 3.6

# Question 2
Now adapt the code in the notebook to perform logistic regression on the iris dataset. It is best to start a new notebook and copy the relevant cells. Since the iris dataset is small, do not bother creating a separate training and test set. Train on the entire dataset. Estimate the number of correctly classfied patterns.

In [10]:
import numpy as np
import pandas as pd

In [11]:
df_iris = pd.read_csv('iris.data', names = ['PL', 'SL', 'PW', 'SW', 'Classification'])
print(df_iris['Classification'].unique())

['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


In [12]:
classification_values = {'Iris-setosa' : 0, 'Iris-virginica' : 1, 'Iris-versicolor' : 2}
df_iris.replace({'Classification' : classification_values}, inplace=True)
df_iris['Bias_Const'] = 1
df_iris.head()

,PL,SL,PW,SW,Classification,Bias_Const
0,5.1,3.5,1.4,0.2,0,1
1,4.9,3.0,1.4,0.2,0,1
2,4.7,3.2,1.3,0.2,0,1
3,4.6,3.1,1.5,0.2,0,1
4,5.0,3.6,1.4,0.2,0,1


In [13]:
design=df_iris[['PL', 'SL', 'PW', 'SW', 'Bias_Const']].to_numpy()
labels=df_iris['Classification'].to_numpy()

target = labels
data = design

print(target.shape)
print(data.shape)

(150,)
(150, 5)


In [14]:
num_variables = 5
num_classes = 3
weights = np.random.normal(0.,1.,size=num_variables*num_classes) # starting with a random set of weights
weights.shape = num_variables,num_classes
print(weights.shape)

(5, 3)


In [25]:
def stableSoftmax(x):
    z = x - np.max(x, axis=-1, keepdims=True)
    numerator = np.exp(z)
    denominator = np.sum(numerator, axis=-1, keepdims=True)
    softmax = numerator / denominator
    return softmax

def update(designbias,weights):
    return stableSoftmax(designbias.dot(weights))

# The gradient is calculated over a single input pattern. This is a 3 x 5 matrix

def gradient(target, weights, designbias,i):
    ms=np.outer(designbias[i],update(designbias,weights)[i] - target[i])
    return ms

def correctlyClassified(labels,designbias,weights):
    outcomes=update(designbias,weights)
    outcomelabels = np.array([ np.argmax(x) for x in outcomes ])
    difftest = outcomelabels - labels
    return np.count_nonzero(difftest==0),outcomes.shape[0],outcomelabels

In [26]:
correctlyClassified(target, data, weights)

(48,
 150,
 array([0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0,
        0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 2, 0, 1, 0,
        0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 0, 0, 1, 0, 0, 0,
        0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0], dtype=int64))

In [45]:
train_targets = pd.get_dummies(df_iris['Classification'])
train_targets.head()

t_target = train_targets.to_numpy()
print(t_target.shape)

(150, 3)


In [49]:
nrdatapoints = design.shape[0]
r = 0.02
for i in range(5000):
    if i%1000 == 0: 
        print(i)
        
    weights -= r*gradient(t_target, weights, design,np.random.randint(0,nrdatapoints))
correctlyClassified(target,data,weights)

0
1000
2000
3000
4000


(147,
 150,
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1,
        1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=int64))

# Question 3:
You should find that logistic regression performs reasonably well. This is remarkable. Logistic regression is a two-layer network, just like the two-layer perceptron network that we considered earlier. Moreover, its decsisions are based on linear combinations of weights and input. Yet, the two-layer perceptron network failed. Why does the logistic regression network succeed?

A logistic regression uses a non-linear activation function to transform the data from the standard x-y space into an exponential space, which allows the data points to become linearly separable

### Actual Answer:
*The softmax output ensures the nodes are not indepdent. The network will be able to learn stosa and virginica, but will also learn that the nodes for virginica and setosa should be ok for a versicolor input pattern. The softmax will then force the versicolor node to respond.*

# Question 4:
Imagine the iris dataset had a forth variety, like versiocolor wedged in between setosa and virginica. Would you expect a two-layer network with softmax output (which now has a four node softmax output) to be able to classify this dataset accurately? Motivate your answer.

### My answer (I read the actual answer before answering here though):
No, it should not perform well as versicolor is derived from not being either setosa or virginica - this definition fits our new 'versiocolor' definition as well, and will mean that it will perform poorly for both versicolor and 'versiocolor'.